<a href="https://colab.research.google.com/github/Lolla-data-analyst/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
df.head(3)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9


# ML Task Framing — Lane 1: Ranking Signal Analysis

## Task Type
Ranking. We want to order pages by how much missed opportunity they
represent, not sort them into fixed categories.

## Target / Proxy
missed_opportunity_score = impressions_90d × (avg_ctr - ctr)

This captures how much visibility (impressions) is being wasted on a
below-average CTR — weighing both the size of the CTR gap and how many
people are affected by it.

## Unit of Analysis
One row = one page's search performance over a 90-day window.

## Success Metric
Precision@50 — if an SEO team member manually reviewed the top 50 pages
ranked by missed_opportunity_score, what percentage would they confirm
are genuine, worthwhile opportunities? A higher percentage means the
ranking is correctly surfacing pages that deserve attention.

## Action
The ranked list tells a content/SEO team member which pages to review
first. For each high-ranked page, they add a clearer call-to-action and
product link, converting existing visibility into clicks.

## Why Ranking Beats a Fixed Rule
A fixed rule (e.g., ctr < avg_ctr * 0.5) only outputs yes/no — it treats
all flagged pages as equally urgent. Ranking captures magnitude: a page
with 100,000 impressions and a small CTR gap represents far more wasted
opportunity than a page with 600 impressions and a worse CTR. This lets
the team prioritize by real-world impact, not just whether a page crosses
an arbitrary threshold.

## Self-check
My missed_opportunity_score assumes CTR gaps matter equally regardless
of content_type or search intent — a transactional page underperforming
might mean something different than an informational page underperforming.
I'm also not accounting for pages with very low impressions but decent
CTR near the 500 threshold, which could shift rankings if the cutoff moves.
A more robust version would segment by intent or normalize by content type
before ranking.

In [2]:
import os, subprocess

REPO_URL = "https://github.com/Lolla-data-analyst/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "CSV not found — check clone worked"
print("Data found, ready to go.")

Working dir: /content/flyrank-ml-internship
Data found, ready to go.


In [4]:
import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

In [5]:
# Calculate average CTR among visible pages (500+ impressions)
visible = df[df["impressions_90d"] >= 500].copy()
avg_ctr = visible["ctr"].mean()

# Calculate missed opportunity score
visible["missed_opportunity_score"] = visible["impressions_90d"] * (avg_ctr - visible["ctr"])

# Show top 10 pages by missed opportunity
visible.sort_values("missed_opportunity_score", ascending=False).head(10)

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,missed_opportunity_score
19636,content_2cb567c3c89b,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,6183.0,39587.0,...,0.10,22.2,5.82,5.64,1.21,excellent,page_3_5,up,23.1,80734.973643
6653,content_5fe46e04994d,client_4e07408562,1900.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.14,4.2,4.23,26.90,0.38,excellent,page_1,down,-44.8,63268.573993
7678,content_8451fc6f034d,client_d029fa3a95,10.0,1.00,HIGH,14.11,keyword article,informational,3528.0,24856.0,...,0.03,2.3,2.02,2.26,2.02,excellent,top_3,up,73.1,63193.834844
3394,content_36ff89c8214e,client_19581e27de,0.0,0.00,LOW,0.00,keyword article,informational,NaN,NaN,...,0.05,7.3,1.68,4.48,0.00,excellent,page_1,stable,0.5,62621.749962
26798,content_b28d1efd668f,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,transactional,6901.0,44938.0,...,0.06,26.2,3.83,6.57,0.96,excellent,page_3_5,stable,-17.2,57954.241837
26844,content_8c19996aa890,client_4e07408562,70.0,0.01,LOW,0.00,keyword article,informational,2895.0,19343.0,...,0.15,2.5,11.73,16.03,0.00,excellent,top_3,down,-44.5,57141.813258
7445,content_c8e9d6ab9013,client_19581e27de,20.0,0.03,LOW,0.00,keyword article,informational,NaN,NaN,...,0.00,9.7,0.00,0.00,0.00,excellent,page_1,down,-43.4,54716.903685
15405,content_a023517539fe,client_6208ef0f77,0.0,0.00,LOW,0.00,keyword article,informational,3530.0,22134.0,...,0.01,85.8,3.40,3.61,0.00,excellent,deep,up,27907.3,53984.224904
6903,content_c84a0ab98e90,client_f369cb89fc,0.0,0.00,LOW,0.00,keyword article,informational,2871.0,19536.0,...,0.03,7.8,3.45,25.00,0.00,excellent,page_1,stable,17.2,51845.165425
26304,content_ff94c9b6b411,client_349c41201b,20.0,0.00,LOW,0.00,keyword article,informational,5375.0,35334.0,...,0.04,27.4,1.15,1.12,1.15,excellent,page_3_5,down,-32.4,50789.043300
